### Finviz and Performance Ratio Merge Pipeline

This notebook merges the cleaned Finviz data with the calculated performance ratios for a specific date.

**Workflow:**

1.  **Prerequisites:** Both the cleaned Finviz data (`{DATE_STR}_df_finviz_stocks_etfs.parquet`) and the calculated performance ratios (`{DATE_STR}_df_perf_ratios_stocks_etfs.parquet`) must exist. The `config.py` file must be up-to-date.
2.  **Load Data:** Loads the two source DataFrames.
3.  **Reconcile & Align:**
    *   Removes any duplicate tickers from both source files.
    *   Identifies the set of common tickers present in both DataFrames.
    *   Filters both DataFrames to only include the common tickers, ensuring a clean merge.
4.  **Merge DataFrames:** Performs a join operation to combine the aligned Finviz and performance ratio data.
5.  **Save & Verify:** Saves the final merged DataFrame and reads it back to confirm success.

### Setup and Configuration

This cell loads all necessary libraries and configuration parameters. It pulls dynamic settings from `config.py` and constructs the file paths for the pipeline.


In [1]:
import sys
from pathlib import Path
import pandas as pd

# --- Project Path Setup ---
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# --- Dynamic Configuration (from config.py) ---
from config import DATE_STR, DEST_DIR

# --- File Path Construction ---
# Use pathlib to create robust, cross-platform paths from the config.
DATA_DIR = Path(DEST_DIR)
FINVIZ_PATH = DATA_DIR / f'{DATE_STR}_df_finviz_stocks_etfs.parquet'
RATIOS_PATH = DATA_DIR / f'{DATE_STR}_df_perf_ratios_stocks_etfs.parquet'
MERGED_DEST_PATH = DATA_DIR / f'{DATE_STR}_df_finviz_n_ratios_stocks_etfs.parquet'
COMMON_TICKERS_PATH = DATA_DIR / f'{DATE_STR}_df_common_tickers_stocks_etfs.parquet'

# --- Notebook Setup ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2500)
%load_ext autoreload
%autoreload 2

# --- Verification ---
print(f"Processing for Date: {DATE_STR}")
print(f"Finviz source: {FINVIZ_PATH}")
print(f"Ratios source: {RATIOS_PATH}")
print(f"Merged destination: {MERGED_DEST_PATH}")

Processing for Date: 2026-07-17
Finviz source: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_stocks_etfs.parquet
Ratios source: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_perf_ratios_stocks_etfs.parquet
Merged destination: c:\Users\ping\Files_win10\python\py311\stocks\data\2026-07-17_df_finviz_n_ratios_stocks_etfs.parquet


### Step 1: Load Source DataFrames

Load the two DataFrames that will be merged.

In [2]:
print("--- Step 1: Loading source data ---")

# Load Finviz data
try:
    df_finviz = pd.read_parquet(FINVIZ_PATH)
    print(f"Successfully loaded Finviz data. Shape: {df_finviz.shape}")
    print(f"df_finviz.info():\n{df_finviz.info()}\n")
except FileNotFoundError:
    print(f"ERROR: Finviz file not found at {FINVIZ_PATH}")
    df_finviz = None

# Load Performance Ratios data
try:
    df_perf_ratios = pd.read_parquet(RATIOS_PATH)
    print(f"Successfully loaded Performance Ratios data. Shape: {df_perf_ratios.shape}")
    print(f"df_perf_ratios.info():\n{df_perf_ratios.info()}\n")
except FileNotFoundError:
    print(f"ERROR: Ratios file not found at {RATIOS_PATH}")
    df_perf_ratios = None

--- Step 1: Loading source data ---
Successfully loaded Finviz data. Shape: (1560, 113)
<class 'pandas.core.frame.DataFrame'>
Index: 1560 entries, NNVDA to GGPIX
Columns: 113 entries, No. to Rank
dtypes: float64(94), int64(3), object(16)
memory usage: 1.4+ MB
df_finviz.info():
None

Successfully loaded Performance Ratios data. Shape: (1545, 24)
<class 'pandas.core.frame.DataFrame'>
Index: 1545 entries, A to ZWS
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Sharpe 3d     1545 non-null   float64
 1   Sortino 3d    1545 non-null   float64
 2   Omega 3d      1545 non-null   float64
 3   Sharpe 5d     1545 non-null   float64
 4   Sortino 5d    1545 non-null   float64
 5   Omega 5d      1545 non-null   float64
 6   Sharpe 10d    1545 non-null   float64
 7   Sortino 10d   1545 non-null   float64
 8   Omega 10d     1545 non-null   float64
 9   Sharpe 15d    1545 non-null   float64
 10  Sortino 15d   1545 non-null   f

In [5]:
print(f"df_finviz.index:\n{df_finviz.index}\n")
print(f"df_perf_ratios.index:\n{df_perf_ratios.index}\n")

df_finviz.index:
Index(['NNVDA', 'AAAPL', 'GGOOGL', 'GGOOG', 'MMSFT', 'AAMZN', 'TTSM', 'AAVGO', 'SSPCX', 'MMETA',
       ...
       'RREET', 'FFDN', 'KKRE', 'SSBIL', 'EEAGG', 'GGPIQ', 'SSJNK', 'SSLYV', 'QQLTY', 'GGPIX'], dtype='object', name='Ticker', length=1560)

df_perf_ratios.index:
Index(['A', 'AA', 'AAL', 'AAON', 'AAPL', 'ABBV', 'ABEV', 'ABNB', 'ABT', 'ACGL',
       ...
       'Z', 'ZBH', 'ZBRA', 'ZG', 'ZION', 'ZM', 'ZS', 'ZTO', 'ZTS', 'ZWS'], dtype='object', length=1545)



In [ ]:
print(f"df_finviz.info():\n{df_finviz.info()}\n")
print(f"df_perf_ratios.info():\n{df_perf_ratios.info()}\n")

<class 'pandas.core.frame.DataFrame'>
Index: 1560 entries, NNVDA to GGPIX
Columns: 113 entries, No. to Rank
dtypes: float64(94), int64(3), object(16)
memory usage: 1.4+ MB
df_finviz.info():
None

<class 'pandas.core.frame.DataFrame'>
Index: 1545 entries, A to ZWS
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Sharpe 3d     1545 non-null   float64
 1   Sortino 3d    1545 non-null   float64
 2   Omega 3d      1545 non-null   float64
 3   Sharpe 5d     1545 non-null   float64
 4   Sortino 5d    1545 non-null   float64
 5   Omega 5d      1545 non-null   float64
 6   Sharpe 10d    1545 non-null   float64
 7   Sortino 10d   1545 non-null   float64
 8   Omega 10d     1545 non-null   float64
 9   Sharpe 15d    1545 non-null   float64
 10  Sortino 15d   1545 non-null   float64
 11  Omega 15d     1545 non-null   float64
 12  Sharpe 30d    1545 non-null   float64
 13  Sortino 30d   1545 non-null   float64
 14  Omega 30d 

### Step 2: Reconcile and Align Ticker Indices

Before merging, we must ensure both DataFrames have identical, clean indices. This involves removing duplicates and filtering both frames to a common set of tickers.


In [3]:
if df_finviz is not None and df_perf_ratios is not None:
    print("\n--- Step 2: Reconciling and aligning ticker indices ---")

    # Part A: Remove any duplicate tickers from each DataFrame, keeping the first entry.
    df_finviz = df_finviz[~df_finviz.index.duplicated(keep="first")]
    df_perf_ratios = df_perf_ratios[~df_perf_ratios.index.duplicated(keep="first")]
    print(f"Deduplicated Finviz shape: {df_finviz.shape}")
    print(f"Deduplicated Ratios shape: {df_perf_ratios.shape}")

    # Part B: Find the intersection of tickers present in both DataFrames.
    common_tickers = df_finviz.index.intersection(df_perf_ratios.index)
    print(f"\nFound {len(common_tickers)} common tickers.")

    # Part C: Filter both DataFrames to only include the common tickers.
    df_finviz_aligned = df_finviz.loc[common_tickers]
    df_perf_ratios_aligned = df_perf_ratios.loc[common_tickers]
    print("Both DataFrames are now filtered to the common set of tickers.")
    print(f"Final aligned Finviz shape: {df_finviz_aligned.shape}")
    print(f"Final aligned Ratios shape: {df_perf_ratios_aligned.shape}")
else:
    print("\nSkipping reconciliation as one or both source files failed to load.")
    df_finviz_aligned, df_perf_ratios_aligned = None, None


--- Step 2: Reconciling and aligning ticker indices ---
Deduplicated Finviz shape: (1560, 113)
Deduplicated Ratios shape: (1545, 24)

Found 14 common tickers.
Both DataFrames are now filtered to the common set of tickers.
Final aligned Finviz shape: (14, 113)
Final aligned Ratios shape: (14, 24)


### Optional Step: Save the List of Common Tickers

This step saves a simple file containing only the list of tickers that were present in both source files. This can be a useful artifact for other analyses.


In [ ]:
if "common_tickers" in locals() and not common_tickers.empty:
    print("\n--- Optional Step: Saving the list of common tickers ---")
    try:
        df_common = pd.DataFrame(index=common_tickers)
        df_common.to_parquet(COMMON_TICKERS_PATH, engine="pyarrow", compression="zstd")
        print(
            f"Successfully saved list of {len(common_tickers)} common tickers to: {COMMON_TICKERS_PATH}"
        )
    except Exception as e:
        print(f"An error occurred while saving the common tickers list: {e}")

### Step 3: Merge Aligned DataFrames

With the DataFrames now cleaned and aligned, perform the join operation. A pre-join check ensures there are no conflicting column names.


In [ ]:
if df_finviz_aligned is not None and df_perf_ratios_aligned is not None:
    print("\n--- Step 3: Merging aligned DataFrames ---")

    # Pre-join check: Ensure there are no overlapping column names.
    duplicate_cols = set(df_finviz_aligned.columns) & set(
        df_perf_ratios_aligned.columns
    )
    if duplicate_cols:
        raise ValueError(f"Merge failed: Duplicate columns found: {duplicate_cols}")

    # Perform the join. Since indices are identical, this is a clean merge.
    df_merged = df_finviz_aligned.join(df_perf_ratios_aligned)
    print("Join successful!")
    df_merged.info()
    display(df_merged.head())
else:
    print("\nSkipping merge step as aligned data is not available.")
    df_merged = None

### Step 4: Save and Verify Final Merged DataFrame

Save the final product and read it back to confirm the entire pipeline was successful.

In [ ]:
if df_merged is not None:
    print("\n--- Step 4: Saving and verifying final merged data ---")
    try:
        df_merged.to_parquet(MERGED_DEST_PATH, engine="pyarrow", compression="zstd")
        print(f"Successfully saved merged DataFrame to: {MERGED_DEST_PATH}")

        # Verification step
        print("\nVerifying saved file...")
        verified_df = pd.read_parquet(MERGED_DEST_PATH)
        print("Verification successful. First 5 rows of saved file:")
        display(verified_df.head())

    except Exception as e:
        print(f"An error occurred during save or verification: {e}")
else:
    print("\nSkipping final save step as merged DataFrame was not created.")